# MPMAvatar — SDS Physics Hypothesis Test
**Lightning AI Studio Notebook**

Pipeline:
1. Clone repo from GitHub
2. Install dependencies + build CUDA extensions
3. Download Wan 2.2 I2V checkpoint
4. Upload / mount your trained Gaussians
5. Run `train_sds_physics.py`

---
**GPU required:** A100 / A10G (40GB+) recommended for Wan 5B

## Cell 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set CUDA arch for custom extension builds
import os
cc = torch.cuda.get_device_capability()
arch = f"{cc[0]}{cc[1]}"
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{cc[0]}.{cc[1]}"
os.environ['FORCE_CUDA'] = '1'
print(f"CUDA arch: sm_{arch}  →  TORCH_CUDA_ARCH_LIST={os.environ['TORCH_CUDA_ARCH_LIST']}")

## Cell 1 — Clone Repo

In [ ]:
import os

# ── CHANGE THESE ──────────────────────────────────────────────────────────────
GITHUB_USER  = "YOUR_GITHUB_USERNAME"       # e.g. "poudelsaroj"
GITHUB_REPO  = "MPMAvatar"                  # repo name (without .git)
GITHUB_TOKEN = ""                           # GitHub Personal Access Token
#   → Settings → Developer settings → Personal access tokens → Fine-grained
#   → Repo access: Contents (read)
# ─────────────────────────────────────────────────────────────────────────────

REPO_DIR = f"/teamspace/studios/this_studio/{GITHUB_REPO}"

if not os.path.exists(REPO_DIR):
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    else:
        clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    print(f"Cloning from github.com/{GITHUB_USER}/{GITHUB_REPO} ...")
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"Repo already at {REPO_DIR}, pulling...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## Cell 2 — Install Python Dependencies

In [ ]:
# Core dependencies — numpy>=1.26 required for Python 3.12+
!pip install -q \
    "numpy>=1.26.0" \
    scipy \
    Pillow \
    tqdm \
    pyyaml \
    wandb \
    einops \
    plyfile \
    trimesh \
    smplx \
    roma \
    jaxtyping \
    mediapy \
    imageio \
    safetensors \
    accelerate

# Warp (MPM solver)
!pip install -q "warp-lang>=1.0.0"

# Transformers + HuggingFace
!pip install -q transformers sentencepiece huggingface_hub

import numpy as np, sys
print(f"numpy {np.__version__} / Python {sys.version.split()[0]}")
print("Done. Run the next cell to install pytorch3d.")

In [ ]:
## Cell 2b — Install pytorch3d (builds from source, ~10-15 min, visible output)
import os
os.environ['FORCE_CUDA'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = '9.0'  # H100

# Install build deps first
!pip install ninja fvcore iopath --quiet

# Build pytorch3d from source with full output so we can see progress
!pip install "git+https://github.com/facebookresearch/pytorch3d.git" \
    --no-build-isolation

try:
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")
except ImportError as e:
    print(f"✗ pytorch3d FAILED: {e}")

## Cell 3 — Build CUDA Extensions

In [ ]:
import os, sys

REPO_DIR = "/teamspace/studios/this_studio/mpm_avatar_vds"
EXT_DIR  = "/teamspace/studios/this_studio/ext"
os.makedirs(EXT_DIR, exist_ok=True)

# ── diff-gaussian-rasterization ───────────────────────────────────────────────
diff_gauss_dir = f"{EXT_DIR}/diff-gaussian-rasterization"
if not os.path.exists(diff_gauss_dir):
    !git clone --recursive \
        https://github.com/slothfulxtx/diff-gaussian-rasterization.git \
        {diff_gauss_dir}

# Patch rasterizer_impl.h: add #include <cstdint> if missing
impl_h = f"{diff_gauss_dir}/cuda_rasterizer/rasterizer_impl.h"
!head -1 {impl_h}
!grep -q "cstdint" {impl_h} || sed -i '1s/^/#include <cstdint>\n/' {impl_h}
print("After patch, first 3 lines:")
!head -3 {impl_h}

# Build using python setup.py directly (no pip isolation)
!cd {diff_gauss_dir} && python setup.py build_ext --inplace 2>&1 | tail -5
!cd {diff_gauss_dir} && python setup.py install 2>&1 | tail -5
print("diff-gaussian-rasterization done")

# ── simple-knn ────────────────────────────────────────────────────────────────
simple_knn_dir = f"{EXT_DIR}/simple-knn"
if not os.path.exists(simple_knn_dir):
    !git clone https://gitlab.inria.fr/bkerbl/simple-knn.git {simple_knn_dir}

!cd {simple_knn_dir} && python setup.py install 2>&1 | tail -5
print("simple-knn done")

# ── Verify ────────────────────────────────────────────────────────────────────
import importlib, subprocess
subprocess.run([sys.executable, "-c", "import diff_gauss; print('✓ diff_gauss OK')"])
subprocess.run([sys.executable, "-c", "import simple_knn; print('✓ simple_knn OK')"])

## Cell 4 — Install Wan 2.2 Repo

In [ ]:
import os, sys

WAN_REPO_DIR = "/teamspace/studios/this_studio/Wan2.2"

if not os.path.exists(WAN_REPO_DIR):
    !git clone https://github.com/Wan-Video/Wan2.2.git {WAN_REPO_DIR}

# Install all deps (librosa needed by wan/__init__.py → speech2video import chain)
!pip install -q easydict ftfy dashscope diffusers accelerate decord tokenizers imageio-ffmpeg librosa opencv-python peft

# flash_attn: download prebuilt wheel directly (avoids cross-device link error)
FLASH_WHL = "flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
FLASH_PATH = f"/tmp/{FLASH_WHL}"
if not os.path.exists(FLASH_PATH):
    FLASH_URL = f"https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/{FLASH_WHL}"
    !wget -q {FLASH_URL} -O {FLASH_PATH}
!pip install -q {FLASH_PATH} --no-deps || echo "flash_attn install failed — continuing without it"

# Install Wan itself
!cd {WAN_REPO_DIR} && pip install -e . --no-build-isolation --no-deps -q

# Pin numpy back to 1.x — some dep above upgrades it to 2.x which breaks wan/pandas/matplotlib
!pip install -q "numpy>=1.26.0,<2.0"

if WAN_REPO_DIR not in sys.path:
    sys.path.insert(0, WAN_REPO_DIR)

import numpy as np
assert np.__version__.startswith("1."), f"numpy must be 1.x, got {np.__version__}"
print(f"numpy {np.__version__} OK")

# Verify
try:
    from wan.modules.model import WanModel
    from wan.modules.vae2_1 import Wan2_1_VAE
    from wan.modules.t5 import T5EncoderModel
    print("✓ Wan modules importable")
except ImportError as e:
    print(f"✗ Wan import failed: {e}")
    print("Run: !pip install <missing_module> then re-run this cell")

## Cell 5 — Download Wan 2.2 I2V Checkpoint

> ⚠️ This is **~40 GB**. Run once, stored persistently in Lightning teamspace.
> Skip if already downloaded.

In [ ]:
import os
from huggingface_hub import snapshot_download, login

# ── Paste your HF token here (never commit this file after filling it in) ─────
HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # <-- replace with your new token
# ─────────────────────────────────────────────────────────────────────────────

login(token=HF_TOKEN, add_to_git_credential=False)

WAN_CKPT_DIR = "/teamspace/studios/this_studio/wan_checkpoints_22"
os.makedirs(WAN_CKPT_DIR, exist_ok=True)

if not os.path.exists(f"{WAN_CKPT_DIR}/low_noise_model"):
    print("Downloading Wan2.2-I2V-A14B (~126GB)...")
    snapshot_download(
        repo_id="Wan-AI/Wan2.2-I2V-A14B",
        local_dir=WAN_CKPT_DIR,
        ignore_patterns=["*.md", "*.txt"],
        token=HF_TOKEN,
    )
    print("Done.")
else:
    print("Already downloaded.")

!ls {WAN_CKPT_DIR}/

## Cell 6 — Pre-download T5 Tokenizer

In [ ]:
T5_CACHE = "/teamspace/studios/this_studio/t5_cache"
os.makedirs(T5_CACHE, exist_ok=True)
os.environ['TRANSFORMERS_CACHE'] = T5_CACHE
os.environ['HF_HOME']            = T5_CACHE

from transformers import AutoTokenizer
print("Downloading umt5-xxl tokenizer (~2GB)...")
AutoTokenizer.from_pretrained('google/umt5-xxl', cache_dir=T5_CACHE)
print("Tokenizer ready.")

## Cell 7 — Upload Trained Gaussians

Upload your trained Gaussian model from Stage 1 (appearance training).

**Option A:** Upload via Lightning Studio file browser (drag and drop)

**Option B:** Upload from local machine using the cell below

Expected structure:
```
/teamspace/studios/this_studio/gaussians/
├── point_cloud/
│   └── iteration_XXXXX/
│       └── point_cloud.ply
├── verts_offset.npy
├── cams.npz
└── shadow_net.pt
```

In [ ]:
import os

!pip install -q gdown

PRETRAINED_DIR = "/teamspace/studios/this_studio/pretrained_models"
os.makedirs(PRETRAINED_DIR, exist_ok=True)

GDRIVE_FOLDER_ID = "1ZylC3f8b3wg6Ae4MkVeHFodJc0OVuJfu"

# --remaining-ok skips folders with >50 files (aomap PNGs) but gets all key files
!gdown --folder https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID} \
    -O {PRETRAINED_DIR} --remaining-ok

!find {PRETRAINED_DIR} -type f | grep -v ".png" | head -40

## Cell 7b — Unzip Uploaded MPMAvatar Assets

If you downloaded the full MPMAvatar assets from Google Drive as a ZIP and uploaded it via the Lightning file browser, run this cell to extract and place everything in the right locations.

**How to upload:**
1. Download the ZIP(s) from the paper's Google Drive to your local machine
2. In Lightning Studio → left sidebar → Files icon → drag & drop the ZIP(s) into `/teamspace/studios/this_studio/uploads/`
3. Set the `ZIP_PATH` variable below and run the cell

The cell handles both ZIPs:
- `MPMAvatar_processed_data.zip` — tracking params, AO maps, split_idx, optimized_weights, smplx_fitted, body models
- The trained Gaussians zip (point_cloud, verts_offset, cams, shadow_net) if packaged separately

In [ ]:
import os, shutil

STUDIO   = "/teamspace/studios/this_studio"
SRC_BASE = f"{STUDIO}/mpm_zip_contents/MPMAvatar_processed_data"
PRETRAINED = f"{STUDIO}/pretrained_models"
REPO_DIR   = f"{STUDIO}/mpm_avatar_vds"

# destination dirs
MODEL_DST    = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
A1S1_DST     = f"{TRACKING_DST}/a1_s1"
REPO_DATA    = f"{REPO_DIR}/data/a1_s1"

for d in [MODEL_DST, A1S1_DST, REPO_DATA]:
    os.makedirs(d, exist_ok=True)

def cp(src, dst):
    if os.path.exists(dst):
        print(f"  · already present: {os.path.basename(dst)}")
        return
    shutil.copy2(src, dst)
    print(f"  ✓ {os.path.basename(dst)}")

# ── Gaussian checkpoint files ─────────────────────────────────────────────────
print("=== Gaussian files ===")
for f in ["point_cloud.ply", "verts_offset.npy", "cams.npz", "shadow_net.pt"]:
    cp(f"{SRC_BASE}/model/a1_s1/point_cloud/timestep_030000/{f}", f"{MODEL_DST}/{f}")

# ── optimized_weights ─────────────────────────────────────────────────────────
print("\n=== Data files ===")
cp(f"{SRC_BASE}/data/a1_s1/optimized_weights.npy", f"{A1S1_DST}/optimized_weights.npy")

# ── Generate missing files using repo scripts ─────────────────────────────────
print("\n=== Generating missing files (split_idx, cam_info, uv_obj, smplx_fitted) ===")
os.chdir(REPO_DIR)

# split_idx.npz is committed to the repo at data/a1_s1/split_idx.npz
repo_split = f"{REPO_DIR}/data/a1_s1/split_idx.npz"
if os.path.exists(repo_split):
    print(f"  · split_idx.npz already in repo")
else:
    print(f"  ✗ split_idx.npz missing from repo — check git clone")

for script in ["gen_cam_info.py", "gen_uv_obj.py", "gen_smplx_params.py", "gen_smplx_fitted.py"]:
    if os.path.exists(f"{REPO_DIR}/{script}"):
        print(f"\n  Running {script} ...")
        ret = os.system(f"python {script}")
        print(f"  {'✓ done' if ret == 0 else '✗ failed (exit ' + str(ret) + ')'}")
    else:
        print(f"  · {script} not found, skipping")

# ── Final check ───────────────────────────────────────────────────────────────
print("\n=== Final asset check ===")
checks = {
    "point_cloud.ply":   f"{MODEL_DST}/point_cloud.ply",
    "cams.npz":          f"{MODEL_DST}/cams.npz",
    "shadow_net.pt":     f"{MODEL_DST}/shadow_net.pt",
    "verts_offset.npy":  f"{MODEL_DST}/verts_offset.npy",
    "params_460.npz":    f"{TRACKING_DST}/params_460.npz",
    "optimized_weights": f"{A1S1_DST}/optimized_weights.npy",
    "split_idx.npz":     f"{REPO_DATA}/split_idx.npz",
    "SMPLX_NEUTRAL.npz": f"{TRACKING_DST}/body_models/smplx/SMPLX_NEUTRAL.npz",
    "TR00_E096.pt":      f"{TRACKING_DST}/body_models/TR00_E096.pt",
    "smplx_fitted":      f"{A1S1_DST}/smplx_fitted",
    "cam_info.json":     f"{A1S1_DST}/cam_info.json",
    "a1s1_uv.obj":       f"{REPO_DATA}/a1s1_uv.obj",
}
all_ok = True
for label, path in checks.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'✓' if ok else '✗ MISSING':<12} {label}")
print()
print("✓ All ready — proceed to Cell 9" if all_ok else "✗ Some missing — see above")

## Cell 7d — Place All Files from Full Dataset TAR

After downloading and extracting `mpmavatar_data.tar.gz`, this cell finds every required file and copies it to the correct location.

```bash
# Download
!pip install -q gdown
!gdown "1d1UXaqhiqOUx1NqggKreGrClldb9dxeR" -O /teamspace/studios/this_studio/mpmavatar_data.tar.gz
# Extract
!tar -xzf /teamspace/studios/this_studio/mpmavatar_data.tar.gz -C /teamspace/studios/this_studio/
```

In [ ]:
import os, shutil, glob

STUDIO      = "/teamspace/studios/this_studio"
PRETRAINED  = f"{STUDIO}/pretrained_models"
REPO_DIR    = f"{STUDIO}/mpm_avatar_vds"

# ── Destination dirs ──────────────────────────────────────────────────────────
MODEL_DST    = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
A1S1_DST     = f"{TRACKING_DST}/a1_s1"
SMPLX_FIT    = f"{A1S1_DST}/smplx_fitted"
BODY_MODELS  = f"{TRACKING_DST}/body_models/smplx"
VPOSER_DST   = f"{TRACKING_DST}/body_models/vposer_v1_0/snapshots/V02_05"
REPO_DATA    = f"{REPO_DIR}/data/a1_s1"

for d in [MODEL_DST, A1S1_DST, SMPLX_FIT, BODY_MODELS, VPOSER_DST, REPO_DATA]:
    os.makedirs(d, exist_ok=True)

# ── Search root(s) — all extracted content ────────────────────────────────────
SEARCH_ROOTS = [
    f"{STUDIO}/mpmavatar_data",          # if tar extracted to this folder
    f"{STUDIO}/MPMAvatar",               # common tar root names
    f"{STUDIO}/data",
    f"{STUDIO}/mpm_zip_contents",
    STUDIO,                              # fallback: search whole studio
]

def find(name, roots=SEARCH_ROOTS, first=True):
    for root in roots:
        if not os.path.isdir(root):
            continue
        hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
        # exclude destinations to avoid self-copy
        hits = [h for h in hits if PRETRAINED not in h and REPO_DIR not in h]
        if hits:
            return hits[0] if first else hits
    return None

def cp(src, dst, label):
    if src is None:
        print(f"  ✗ NOT FOUND  {label}")
        return False
    if os.path.exists(dst):
        print(f"  · exists     {label}")
        return True
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  ✓ copied     {label}")
    return True

def cpdir(src_dir, dst_dir, label):
    if src_dir is None:
        print(f"  ✗ NOT FOUND  {label}")
        return 0
    files = glob.glob(f"{src_dir}/*")
    n = 0
    for f in files:
        dst = os.path.join(dst_dir, os.path.basename(f))
        if not os.path.exists(dst):
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(f, dst)
            n += 1
    existing = len(glob.glob(f"{dst_dir}/*"))
    print(f"  ✓ {label}: {n} new copied ({existing} total in dst)")
    return n

print("=" * 60)
print("Gaussian checkpoint files")
print("=" * 60)
cp(find("point_cloud.ply"),  f"{MODEL_DST}/point_cloud.ply",  "point_cloud.ply")
cp(find("verts_offset.npy"), f"{MODEL_DST}/verts_offset.npy", "verts_offset.npy")
cp(find("cams.npz"),         f"{MODEL_DST}/cams.npz",         "cams.npz")
cp(find("shadow_net.pt"),    f"{MODEL_DST}/shadow_net.pt",    "shadow_net.pt")

print()
print("=" * 60)
print("Tracking / physics params")
print("=" * 60)
cp(find("optimized_weights.npy"), f"{A1S1_DST}/optimized_weights.npy", "optimized_weights.npy")
cp(find("cam_info.json"),         f"{TRACKING_DST}/cam_info.json",     "cam_info.json")
cp(find("a1s1_uv.obj"),           f"{REPO_DATA}/a1s1_uv.obj",          "a1s1_uv.obj")
cp(find("split_idx.npz"),         f"{REPO_DATA}/split_idx.npz",        "split_idx.npz")
cp(find("cloth_sim.obj"),         f"{REPO_DATA}/cloth_sim.obj",        "cloth_sim.obj")

# params_*.npz
params_hits = find("params_460.npz", first=False) or []
if params_hits:
    params_src_dir = os.path.dirname(params_hits[0]) if params_hits else None
    cpdir(params_src_dir, TRACKING_DST, "params_*.npz (tracking)")
else:
    # try finding any params_ file
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root): continue
        hits = [h for h in glob.glob(f"{root}/**/params_*.npz", recursive=True)
                if PRETRAINED not in h and REPO_DIR not in h]
        if hits:
            params_src_dir = os.path.dirname(hits[0])
            cpdir(params_src_dir, TRACKING_DST, "params_*.npz (tracking)")
            break
    else:
        print("  ✗ NOT FOUND  params_*.npz")

# AO maps
for root in SEARCH_ROOTS:
    if not os.path.isdir(root): continue
    ao_hits = [h for h in glob.glob(f"{root}/**/ao_*.png", recursive=True)
               if PRETRAINED not in h and REPO_DIR not in h]
    if ao_hits:
        cpdir(os.path.dirname(ao_hits[0]), TRACKING_DST, "ao_*.png (AO maps)")
        break
else:
    print("  ✗ NOT FOUND  ao_*.png")

print()
print("=" * 60)
print("smplx_fitted frames")
print("=" * 60)
for root in SEARCH_ROOTS:
    if not os.path.isdir(root): continue
    sf_hits = [h for h in glob.glob(f"{root}/**/smplx_fitted/*", recursive=True)
               if PRETRAINED not in h and REPO_DIR not in h
               and os.path.isfile(h)]
    if sf_hits:
        sf_src = os.path.dirname(sf_hits[0])
        cpdir(sf_src, SMPLX_FIT, "smplx_fitted frames")
        break
else:
    print("  ✗ NOT FOUND  smplx_fitted/")

print()
print("=" * 60)
print("Body models")
print("=" * 60)
for name in ["SMPLX_NEUTRAL.npz", "SMPLX_MALE.npz", "SMPLX_FEMALE.npz"]:
    cp(find(name), f"{BODY_MODELS}/{name}", name)
cp(find("TR00_E096.pt"), f"{VPOSER_DST}/TR00_E096.pt", "TR00_E096.pt (VPoser)")

print()
print("=" * 60)
print("Summary check")
print("=" * 60)
required = {
    "point_cloud.ply":      f"{MODEL_DST}/point_cloud.ply",
    "verts_offset.npy":     f"{MODEL_DST}/verts_offset.npy",
    "cams.npz":             f"{MODEL_DST}/cams.npz",
    "shadow_net.pt":        f"{MODEL_DST}/shadow_net.pt",
    "optimized_weights":    f"{A1S1_DST}/optimized_weights.npy",
    "split_idx.npz":        f"{REPO_DATA}/split_idx.npz",
    "params_460.npz":       f"{TRACKING_DST}/params_460.npz",
    "SMPLX_NEUTRAL.npz":    f"{BODY_MODELS}/SMPLX_NEUTRAL.npz",
    "TR00_E096.pt":         f"{VPOSER_DST}/TR00_E096.pt",
    "smplx_fitted/000460":  f"{SMPLX_FIT}/000460.pth",
}
all_ok = True
for name, path in required.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'✓' if ok else '✗'} {name}")
print()
print("✓ All required files present!" if all_ok else "✗ Some files missing — check above")


In [ ]:
  !pip install -q human-body-prior


In [ ]:
EXTRACT_DIR = "/teamspace/studios/this_studio/mpm_zip_contents"

targets = [
    "point_cloud.ply",
    "verts_offset.npy",
    "cams.npz",
    "shadow_net.pt",
    "split_idx.npz",
    "optimized_weights.npy",
    "smplx_fitted",
    "cam_info.json",
    "a1s1_uv.obj",
    "SMPLX_NEUTRAL.npz",
    "TR00_E096.pt",
]

import glob, os
print(f"Searching in: {EXTRACT_DIR}\n")
for name in targets:
    hits = glob.glob(f"{EXTRACT_DIR}/**/{name}", recursive=True)
    if hits:
        for h in hits:
            print(f"  FOUND  {name:30s}  →  {os.path.relpath(h, EXTRACT_DIR)}")
    else:
        print(f"  MISSING {name}")

In [ ]:
import zipfile, glob, os

ZIP_PATH    = "/teamspace/studios/this_studio/MPMAvatar_processed_data.zip"
EXTRACT_DIR = "/teamspace/studios/this_studio/mpm_zip_contents"

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

all_files = sorted([
    f for f in glob.glob(f"{EXTRACT_DIR}/**/*", recursive=True)
    if os.path.isfile(f)
])
print(f"{len(all_files)} files extracted:\n")
for f in all_files:
    print(os.path.relpath(f, EXTRACT_DIR))

In [ ]:
import os, zipfile, shutil, glob

STUDIO      = "/teamspace/studios/this_studio"
ZIP_PATH    = f"{STUDIO}/MPMAvatar_processed_data.zip"   # <-- change if your zip is elsewhere
EXTRACT_DIR = f"{STUDIO}/mpm_zip_contents"

# ── Step 1: Unzip ─────────────────────────────────────────────────────────────
if not os.path.exists(ZIP_PATH):
    print(f"✗ ZIP not found: {ZIP_PATH}")
    print("  Set ZIP_PATH to wherever you uploaded the file.")
else:
    print(f"Extracting {os.path.basename(ZIP_PATH)} ...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Done.\n")

    # ── Step 2: List every file inside ────────────────────────────────────────
    all_files = sorted([
        f for f in glob.glob(f"{EXTRACT_DIR}/**/*", recursive=True)
        if os.path.isfile(f)
    ])
    print(f"=== {len(all_files)} files extracted ===")
    for f in all_files:
        print(f"  {os.path.relpath(f, EXTRACT_DIR)}")

    # ── Step 3: Copy every known asset to its destination ─────────────────────
    PRETRAINED   = f"{STUDIO}/pretrained_models"
    TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
    MODEL_DST    = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
    BODY_DST     = f"{TRACKING_DST}/body_models"
    SMPLX_DST    = f"{BODY_DST}/smplx"
    A1S1_DST     = f"{TRACKING_DST}/a1_s1"
    REPO_DATA    = f"{STUDIO}/mpm_avatar_vds/data/a1_s1"

    for d in [TRACKING_DST, MODEL_DST, SMPLX_DST, A1S1_DST, REPO_DATA,
              f"{TRACKING_DST}/aomap"]:
        os.makedirs(d, exist_ok=True)

    def cp(src, dst):
        if os.path.exists(dst):
            print(f"  · skip (exists): {os.path.basename(dst)}")
            return
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f"  ✓ {os.path.basename(dst)}")

    print("\n=== Copying files ===")
    for f in all_files:
        name = os.path.basename(f)
        rel  = os.path.relpath(f, EXTRACT_DIR)

        # Gaussian checkpoint files
        if name in ("point_cloud.ply", "verts_offset.npy", "cams.npz", "shadow_net.pt"):
            cp(f, f"{MODEL_DST}/{name}")

        # Tracking params
        elif name.startswith("params_") and name.endswith(".npz"):
            cp(f, f"{TRACKING_DST}/{name}")

        # AO maps
        elif name.startswith("mesh_cloth_") and name.endswith(".png"):
            cp(f, f"{TRACKING_DST}/aomap/{name}")

        # optimized_weights
        elif name == "optimized_weights.npy":
            cp(f, f"{A1S1_DST}/{name}")

        # split_idx
        elif name == "split_idx.npz":
            cp(f, f"{REPO_DATA}/{name}")

        # SMPLX body models
        elif name.startswith("SMPLX_") and name.endswith(".npz"):
            cp(f, f"{SMPLX_DST}/{name}")

        # VPoser
        elif name == "TR00_E096.pt" or ("vposer" in rel.lower() and name.endswith(".pt")):
            cp(f, f"{BODY_DST}/TR00_E096.pt")

        # smplx_fitted per-frame files
        elif "smplx_fitted" in rel:
            frame_rel = rel[rel.index("smplx_fitted") + len("smplx_fitted/"):]
            cp(f, f"{A1S1_DST}/smplx_fitted/{frame_rel}")

        # cam_info
        elif name == "cam_info.json":
            cp(f, f"{A1S1_DST}/{name}")

        # UV obj
        elif name == "a1s1_uv.obj":
            cp(f, f"{REPO_DATA}/{name}")

    # ── Step 4: Final summary ──────────────────────────────────────────────────
    print("\n=== Asset check ===")
    checks = {
        "point_cloud.ply":   f"{MODEL_DST}/point_cloud.ply",
        "cams.npz":          f"{MODEL_DST}/cams.npz",
        "shadow_net.pt":     f"{MODEL_DST}/shadow_net.pt",
        "verts_offset.npy":  f"{MODEL_DST}/verts_offset.npy",
        "params_460.npz":    f"{TRACKING_DST}/params_460.npz",
        "optimized_weights": f"{A1S1_DST}/optimized_weights.npy",
        "split_idx.npz":     f"{REPO_DATA}/split_idx.npz",
        "SMPLX_NEUTRAL.npz": f"{SMPLX_DST}/SMPLX_NEUTRAL.npz",
        "TR00_E096.pt":      f"{BODY_DST}/TR00_E096.pt",
        "smplx_fitted":      f"{A1S1_DST}/smplx_fitted",
        "cam_info.json":     f"{A1S1_DST}/cam_info.json",
        "a1s1_uv.obj":       f"{REPO_DATA}/a1s1_uv.obj",
    }
    all_ok = True
    for label, path in checks.items():
        ok = os.path.exists(path)
        if not ok: all_ok = False
        print(f"  {'✓' if ok else '✗ MISSING':<12} {label}")
    print()
    print("✓ All good — proceed to Cell 9" if all_ok else "✗ Some missing — check the file list above")

## Cell 7c — Extract VPoser + SMPLX Body Models

Extracts `vposer_v1_0.zip` and `models_smplx_v1_1.zip` into the correct locations.

In [ ]:
import os, shutil, glob

STUDIO       = "/teamspace/studios/this_studio"
EXTRACT_DIR  = f"{STUDIO}/mpm_extracted"   # where the ZIP was extracted
PRETRAINED   = f"{STUDIO}/pretrained_models"
MODEL_DST    = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
A1S1_DST     = f"{TRACKING_DST}/a1_s1"
REPO_DIR     = f"{STUDIO}/mpm_avatar_vds"
REPO_DATA    = f"{REPO_DIR}/data/a1_s1"

os.makedirs(MODEL_DST,   exist_ok=True)
os.makedirs(A1S1_DST,    exist_ok=True)
os.makedirs(REPO_DATA,   exist_ok=True)

def _cp(src, dst, label=""):
    if not os.path.exists(src): return False
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  ✓ {label or os.path.basename(dst)}")
    else:
        print(f"  · already present: {label or os.path.basename(dst)}")
    return True

# ── Search extracted ZIP for Gaussian files ───────────────────────────────────
print("=== Searching extracted ZIP for Gaussian + missing files ===\n")

gaussian_files = ["point_cloud.ply", "verts_offset.npy", "cams.npz", "shadow_net.pt"]
for fname in gaussian_files:
    found = glob.glob(f"{EXTRACT_DIR}/**/{fname}", recursive=True)
    if found:
        _cp(found[0], os.path.join(MODEL_DST, fname), fname)
    else:
        print(f"  ✗ {fname} not found in extracted ZIP")

# ── split_idx.npz ─────────────────────────────────────────────────────────────
for f in glob.glob(f"{EXTRACT_DIR}/**/split_idx.npz", recursive=True):
    _cp(f, f"{REPO_DATA}/split_idx.npz", "split_idx.npz")
    break
else:
    # also check if it's already committed to the repo
    repo_split = f"{REPO_DIR}/data/a1_s1/split_idx.npz"
    if os.path.exists(repo_split):
        print(f"  · split_idx.npz already in repo at {repo_split}")
    else:
        print(f"  ✗ split_idx.npz not found anywhere")

# ── smplx_fitted ─────────────────────────────────────────────────────────────
smplx_src = next(
    (d for d in glob.glob(f"{EXTRACT_DIR}/**/smplx_fitted", recursive=True)
     if os.path.isdir(d)), None
)
if smplx_src:
    dst_dir = f"{A1S1_DST}/smplx_fitted"
    files = [f for f in glob.glob(f"{smplx_src}/**/*", recursive=True) if os.path.isfile(f)]
    copied = 0
    for f in files:
        rel = os.path.relpath(f, smplx_src)
        dst_f = os.path.join(dst_dir, rel)
        if not os.path.exists(dst_f):
            os.makedirs(os.path.dirname(dst_f), exist_ok=True)
            shutil.copy2(f, dst_f)
            copied += 1
    print(f"  ✓ smplx_fitted: {copied} files copied → {dst_dir}")
else:
    print(f"  ✗ smplx_fitted not found in extracted ZIP")

# ── cam_info.json ─────────────────────────────────────────────────────────────
for f in glob.glob(f"{EXTRACT_DIR}/**/cam_info.json", recursive=True):
    _cp(f, f"{A1S1_DST}/cam_info.json", "cam_info.json")
    break
else:
    print(f"  ✗ cam_info.json not in ZIP (will need to run gen_cam_info.py)")

# ── a1s1_uv.obj ──────────────────────────────────────────────────────────────
for f in glob.glob(f"{EXTRACT_DIR}/**/a1s1_uv.obj", recursive=True):
    _cp(f, f"{REPO_DATA}/a1s1_uv.obj", "a1s1_uv.obj")
    break
else:
    print(f"  ✗ a1s1_uv.obj not in ZIP (will need to run gen_uv_obj.py)")

# ── If Gaussians still missing: show full tree to diagnose ───────────────────
still_missing = [f for f in gaussian_files
                 if not os.path.exists(os.path.join(MODEL_DST, f))]
if still_missing:
    print(f"\n  Still missing: {still_missing}")
    print(f"  Showing all files in extracted ZIP (first 60):")
    all_files = sorted(glob.glob(f"{EXTRACT_DIR}/**/*", recursive=True))
    for f in all_files[:60]:
        if os.path.isfile(f):
            print(f"    {os.path.relpath(f, EXTRACT_DIR)}")
    if len(all_files) > 60:
        print(f"    ... and {len(all_files)-60} more")

# ── Final check ───────────────────────────────────────────────────────────────
print("\n=== Final check ===")
checks = {
    "point_cloud.ply":   f"{MODEL_DST}/point_cloud.ply",
    "cams.npz":          f"{MODEL_DST}/cams.npz",
    "shadow_net.pt":     f"{MODEL_DST}/shadow_net.pt",
    "verts_offset.npy":  f"{MODEL_DST}/verts_offset.npy",
    "split_idx.npz":     f"{REPO_DATA}/split_idx.npz",
    "smplx_fitted":      f"{A1S1_DST}/smplx_fitted",
    "cam_info.json":     f"{A1S1_DST}/cam_info.json",
    "a1s1_uv.obj":       f"{REPO_DATA}/a1s1_uv.obj",
}
for label, path in checks.items():
    print(f"  {'✓' if os.path.exists(path) else '✗ MISSING':<12} {label}")

In [ ]:
import os, zipfile, shutil, glob

STUDIO       = "/teamspace/studios/this_studio"
PRETRAINED   = f"{STUDIO}/pretrained_models"
TRACKING_DST = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
BODY_DST     = f"{TRACKING_DST}/body_models"
SMPLX_DST    = f"{BODY_DST}/smplx"

os.makedirs(SMPLX_DST, exist_ok=True)
os.makedirs(BODY_DST,  exist_ok=True)

VPOSER_ZIP = "/teamspace/studios/this_studio/vposer_v1_0.zip"
SMPLX_ZIP  = "/teamspace/studios/this_studio/models_smplx_v1_1.zip"

# ── Helper ────────────────────────────────────────────────────────────────────
def _cp(src, dst, label=""):
    if not os.path.exists(src):
        return False
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  ✓ {label or os.path.basename(dst)}")
    else:
        print(f"  · already present: {label or os.path.basename(dst)}")
    return True

# ── Extract VPoser ─────────────────────────────────────────────────────────────
# vposer_v1_0.zip contains the VPoser v1.0 weights.
# We need TR00_E096.pt (or any .pt) placed at BODY_DST/TR00_E096.pt
if os.path.exists(VPOSER_ZIP):
    print(f"=== Extracting {os.path.basename(VPOSER_ZIP)} ===")
    VP_EXTRACT = f"{STUDIO}/vposer_extracted"
    os.makedirs(VP_EXTRACT, exist_ok=True)

    with zipfile.ZipFile(VPOSER_ZIP, "r") as zf:
        print(f"  {len(zf.namelist())} entries")
        zf.extractall(VP_EXTRACT)

    # Show what came out
    all_files = [f for f in glob.glob(f"{VP_EXTRACT}/**/*", recursive=True) if os.path.isfile(f)]
    print(f"  Extracted {len(all_files)} files:")
    for f in all_files[:30]:
        print(f"    {os.path.relpath(f, VP_EXTRACT)}")

    # Copy any .pt file as TR00_E096.pt (VPoser checkpoint)
    pt_files = glob.glob(f"{VP_EXTRACT}/**/*.pt", recursive=True)
    if pt_files:
        # Prefer file named TR00_E096.pt if present, else take the first .pt
        match = next((f for f in pt_files if "TR00_E096" in f), pt_files[0])
        _cp(match, f"{BODY_DST}/TR00_E096.pt", "TR00_E096.pt (VPoser)")
    else:
        print("  ✗ No .pt file found in VPoser ZIP — listing all files:")
        for f in all_files:
            print(f"    {os.path.relpath(f, VP_EXTRACT)}")
else:
    print(f"✗ Not found: {VPOSER_ZIP}")

# ── Extract SMPLX ──────────────────────────────────────────────────────────────
# models_smplx_v1_1.zip contains SMPLX_NEUTRAL.npz (and MALE/FEMALE variants).
# Place all *.npz files at BODY_DST/smplx/
if os.path.exists(SMPLX_ZIP):
    print(f"\n=== Extracting {os.path.basename(SMPLX_ZIP)} ===")
    SMPLX_EXTRACT = f"{STUDIO}/smplx_extracted"
    os.makedirs(SMPLX_EXTRACT, exist_ok=True)

    with zipfile.ZipFile(SMPLX_ZIP, "r") as zf:
        print(f"  {len(zf.namelist())} entries")
        zf.extractall(SMPLX_EXTRACT)

    npz_files = glob.glob(f"{SMPLX_EXTRACT}/**/*.npz", recursive=True)
    print(f"  Found {len(npz_files)} .npz files:")
    for f in npz_files:
        dst = os.path.join(SMPLX_DST, os.path.basename(f))
        _cp(f, dst)
else:
    print(f"✗ Not found: {SMPLX_ZIP}")

# ── Summary ────────────────────────────────────────────────────────────────────
print("\n=== Body model check ===")
checks = {
    "SMPLX_NEUTRAL.npz": f"{SMPLX_DST}/SMPLX_NEUTRAL.npz",
    "SMPLX_MALE.npz":    f"{SMPLX_DST}/SMPLX_MALE.npz",
    "SMPLX_FEMALE.npz":  f"{SMPLX_DST}/SMPLX_FEMALE.npz",
    "TR00_E096.pt":      f"{BODY_DST}/TR00_E096.pt",
}
for label, path in checks.items():
    print(f"  {'✓' if os.path.exists(path) else '✗ MISSING':<12} {label}")

In [ ]:
import os, zipfile, shutil, glob

# ── CONFIGURE: set the path(s) to your uploaded ZIP file(s) ──────────────────
#
#   You can pass one or two ZIPs:
#     ZIP_PROCESSED  — MPMAvatar_processed_data.zip  (tracking, split_idx, body models …)
#     ZIP_GAUSSIANS  — Gaussians zip                  (point_cloud, shadow_net, cams …)
#                      Leave empty ("") if you used the gdown cell above instead.
#
ZIP_PROCESSED = "/teamspace/studios/this_studio/uploads/MPMAvatar_processed_data.zip"
ZIP_GAUSSIANS = ""   # e.g. "/teamspace/studios/this_studio/uploads/gaussians.zip"
# ─────────────────────────────────────────────────────────────────────────────

STUDIO        = "/teamspace/studios/this_studio"
PRETRAINED    = f"{STUDIO}/pretrained_models"
REPO_DIR      = f"{STUDIO}/mpm_avatar_vds"

# Destinations (must match Cell 9 config)
TRACKING_DST      = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
MODEL_DST         = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
BODY_MODELS_DST   = f"{TRACKING_DST}/body_models"
SMPLX_DST         = f"{BODY_MODELS_DST}/smplx"
A1S1_DST          = f"{TRACKING_DST}/a1_s1"
REPO_DATA_DST     = f"{REPO_DIR}/data/a1_s1"

for d in [TRACKING_DST, MODEL_DST, SMPLX_DST, A1S1_DST, REPO_DATA_DST]:
    os.makedirs(d, exist_ok=True)

# ── Helper: find the root inside a ZIP (handles single-folder wrappers) ───────
def _zip_root(zf: zipfile.ZipFile) -> str:
    """Return the single top-level directory inside the ZIP, or '' if multiple."""
    tops = {n.split("/")[0] for n in zf.namelist() if n.strip("/")}
    return tops.pop() if len(tops) == 1 else ""

# ── Helper: copy a file only if destination doesn't already exist ─────────────
def _cp(src: str, dst: str, label: str = "") -> bool:
    if not os.path.exists(src):
        return False
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  ✓ {label or os.path.basename(dst)}")
    else:
        print(f"  · already present: {label or os.path.basename(dst)}")
    return True

# ── Extract + place MPMAvatar_processed_data.zip ─────────────────────────────
if ZIP_PROCESSED and os.path.exists(ZIP_PROCESSED):
    print(f"\n=== Extracting {os.path.basename(ZIP_PROCESSED)} ===")
    EXTRACT_DIR = f"{STUDIO}/mpm_extracted"
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(ZIP_PROCESSED, "r") as zf:
        root = _zip_root(zf)
        print(f"  ZIP root folder: '{root}'  ({len(zf.namelist())} entries)")
        zf.extractall(EXTRACT_DIR)

    # The ZIP may extract as:
    #   EXTRACT_DIR/<root>/...
    # or with double nesting (Google Drive packaging):
    #   EXTRACT_DIR/<root>/<root>/...
    # Walk down until we find the actual content root.
    src_base = os.path.join(EXTRACT_DIR, root) if root else EXTRACT_DIR
    if os.path.isdir(os.path.join(src_base, root)):
        src_base = os.path.join(src_base, root)   # unwrap double nesting
    print(f"  Content root: {src_base}")

    # ── 1. Tracking params_*.npz + aomap PNGs ─────────────────────────────
    params_files = glob.glob(f"{src_base}/**/params_*.npz", recursive=True)
    print(f"\n  Tracking params: {len(params_files)} files")
    for f in params_files:
        _cp(f, os.path.join(TRACKING_DST, os.path.basename(f)))

    aomap_dir_src = next(
        (d for d in glob.glob(f"{src_base}/**/aomap", recursive=True)
         if os.path.isdir(d)), None
    )
    if aomap_dir_src:
        aomap_dir_dst = os.path.join(TRACKING_DST, "aomap")
        os.makedirs(aomap_dir_dst, exist_ok=True)
        pngs = glob.glob(f"{aomap_dir_src}/*.png")
        print(f"  AO maps: {len(pngs)} PNGs")
        for f in pngs:
            _cp(f, os.path.join(aomap_dir_dst, os.path.basename(f)))
    else:
        print("  AO maps: not found in ZIP")

    # ── 2. split_idx.npz ──────────────────────────────────────────────────
    print()
    splits = glob.glob(f"{src_base}/**/split_idx.npz", recursive=True)
    if splits:
        _cp(splits[0], os.path.join(REPO_DATA_DST, "split_idx.npz"), "split_idx.npz → repo data")
    else:
        print("  ✗ split_idx.npz not found in ZIP")

    # ── 3. optimized_weights.npy ──────────────────────────────────────────
    wts = glob.glob(f"{src_base}/**/optimized_weights.npy", recursive=True)
    if wts:
        _cp(wts[0], os.path.join(A1S1_DST, "optimized_weights.npy"), "optimized_weights.npy")
    else:
        print("  ✗ optimized_weights.npy not found in ZIP")

    # ── 4. smplx_fitted (per-frame .pth / .obj files) ────────────────────
    smplx_fitted_src = next(
        (d for d in glob.glob(f"{src_base}/**/smplx_fitted", recursive=True)
         if os.path.isdir(d)), None
    )
    if smplx_fitted_src:
        smplx_fitted_dst = os.path.join(A1S1_DST, "smplx_fitted")
        all_frames = [
            f for f in glob.glob(f"{smplx_fitted_src}/**/*", recursive=True)
            if os.path.isfile(f)
        ]
        print(f"\n  smplx_fitted: {len(all_frames)} files")
        copied = 0
        for f in all_frames:
            rel = os.path.relpath(f, smplx_fitted_src)
            dst_f = os.path.join(smplx_fitted_dst, rel)
            if not os.path.exists(dst_f):
                os.makedirs(os.path.dirname(dst_f), exist_ok=True)
                shutil.copy2(f, dst_f)
                copied += 1
        print(f"  smplx_fitted: copied {copied} new files → {smplx_fitted_dst}")
    else:
        print("  smplx_fitted: not found in ZIP")

    # ── 5. Body models (SMPLX npz + VPoser) ──────────────────────────────
    print()
    for npz in glob.glob(f"{src_base}/**/SMPLX_*.npz", recursive=True):
        _cp(npz, os.path.join(SMPLX_DST, os.path.basename(npz)))
    for npz in glob.glob(f"{src_base}/**/smplx/*.npz", recursive=True):
        _cp(npz, os.path.join(SMPLX_DST, os.path.basename(npz)))
    vposer_pts = glob.glob(f"{src_base}/**/TR00_E096.pt", recursive=True)
    if vposer_pts:
        _cp(vposer_pts[0], os.path.join(BODY_MODELS_DST, "TR00_E096.pt"), "TR00_E096.pt (VPoser)")
    else:
        print("  VPoser TR00_E096.pt: not found in ZIP")

    # ── 6. cam_info.json ──────────────────────────────────────────────────
    for f in glob.glob(f"{src_base}/**/cam_info.json", recursive=True):
        _cp(f, os.path.join(A1S1_DST, "cam_info.json"), "cam_info.json")
        break

    # ── 7. a1s1_uv.obj ────────────────────────────────────────────────────
    for f in glob.glob(f"{src_base}/**/a1s1_uv.obj", recursive=True):
        _cp(f, os.path.join(REPO_DATA_DST, "a1s1_uv.obj"), "a1s1_uv.obj")
        break

else:
    if ZIP_PROCESSED:
        print(f"✗ File not found: {ZIP_PROCESSED}")
        print("  Upload the ZIP via Lightning Studio file browser first.")
    else:
        print("ZIP_PROCESSED not set — skipping processed data extraction.")

# ── Extract + place Gaussians ZIP (optional) ─────────────────────────────────
if ZIP_GAUSSIANS and os.path.exists(ZIP_GAUSSIANS):
    print(f"\n=== Extracting {os.path.basename(ZIP_GAUSSIANS)} ===")
    GAUSS_EXTRACT = f"{STUDIO}/gauss_extracted"
    os.makedirs(GAUSS_EXTRACT, exist_ok=True)

    with zipfile.ZipFile(ZIP_GAUSSIANS, "r") as zf:
        root = _zip_root(zf)
        print(f"  ZIP root: '{root}'  ({len(zf.namelist())} entries)")
        zf.extractall(GAUSS_EXTRACT)

    src_base = os.path.join(GAUSS_EXTRACT, root) if root else GAUSS_EXTRACT
    if os.path.isdir(os.path.join(src_base, root)):
        src_base = os.path.join(src_base, root)

    # Gaussian assets
    for fname in ["point_cloud.ply", "verts_offset.npy", "cams.npz", "shadow_net.pt"]:
        found = glob.glob(f"{src_base}/**/{fname}", recursive=True)
        if found:
            _cp(found[0], os.path.join(MODEL_DST, fname), fname)
        else:
            print(f"  ✗ {fname} not found in ZIP")
elif ZIP_GAUSSIANS:
    print(f"✗ File not found: {ZIP_GAUSSIANS}")

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n=== File check after extraction ===")
important = {
    "params_460.npz":     f"{TRACKING_DST}/params_460.npz",
    "point_cloud.ply":    f"{MODEL_DST}/point_cloud.ply",
    "cams.npz":           f"{MODEL_DST}/cams.npz",
    "shadow_net.pt":      f"{MODEL_DST}/shadow_net.pt",
    "verts_offset.npy":   f"{MODEL_DST}/verts_offset.npy",
    "split_idx.npz":      f"{REPO_DATA_DST}/split_idx.npz",
    "optimized_weights":  f"{A1S1_DST}/optimized_weights.npy",
    "SMPLX_NEUTRAL.npz":  f"{SMPLX_DST}/SMPLX_NEUTRAL.npz",
    "VPoser TR00_E096.pt":f"{BODY_MODELS_DST}/TR00_E096.pt",
    "smplx_fitted dir":   f"{A1S1_DST}/smplx_fitted",
    "cam_info.json":      f"{A1S1_DST}/cam_info.json",
    "a1s1_uv.obj":        f"{REPO_DATA_DST}/a1s1_uv.obj",
}
all_ok = True
for label, path in important.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"  {'✓' if ok else '✗ MISSING':<12} {label}")
print()
print("✓ All assets in place — proceed to Cell 9 (Config)" if all_ok
      else "✗ Some assets missing — check above and re-run or copy manually")

## Cell 8 — Upload Dataset Files

Upload your `split_idx.npz` and SMPLX fitted params.

Expected structure:
```
/teamspace/studios/this_studio/data/
├── split_idx.npz
├── a1_s1/
│   └── smplx_fitted/
│       ├── 000460.pth
│       ├── 000461.pth
│       └── ...
└── body_models/
    └── smplx/
```

In [ ]:
DATA_DIR = "/teamspace/studios/this_studio/data"

checks = [
    f"{DATA_DIR}/split_idx.npz",
]
for f in checks:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")

## Cell 9 — Config

Set all paths here. Everything else uses these variables.

In [ ]:
import os

REPO_DIR    = "/teamspace/studios/this_studio/mpm_avatar_vds"
PRETRAINED  = "/teamspace/studios/this_studio/pretrained_models"

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAINED_MODEL_PATH = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
MODEL_PATH         = TRAINED_MODEL_PATH
DATASET_DIR        = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
SPLIT_IDX_PATH     = f"{REPO_DIR}/data/a1_s1/split_idx.npz"
OUTPUT_DIR         = "/teamspace/studios/this_studio/output"
WAN_CKPT_DIR       = "/teamspace/studios/this_studio/wan_checkpoints_22"   # Wan2.2-I2V-A14B
WAN_REPO_ROOT      = "/teamspace/studios/this_studio/Wan2.2"
SDS_CFG            = f"{REPO_DIR}/bridge_sds/configs/sds_test.yaml"
T5_CACHE           = "/teamspace/studios/this_studio/t5_cache"

# ── Dataset config ─────────────────────────────────────────────────────────────
ACTOR              = 1
SEQUENCE           = 1
TRAIN_FRAME_START  = 460
TRAIN_FRAME_NUM    = 10
VERTS_START_IDX    = 460

# ── Training ───────────────────────────────────────────────────────────────────
ITERATIONS         = 100
WANDB_PROJECT      = "MPMAvatar_SDS"
USE_WANDB          = False
WANDB_API_KEY      = ""

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(REPO_DIR)

# ── W&B login (set WANDB_API_KEY above or leave "" to skip) ──────────────────
if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print("W&B logged in")

# ── Generate split_idx.npz if not present ─────────────────────────────────────
if not os.path.exists(SPLIT_IDX_PATH):
    print("Generating split_idx.npz ...")
    import subprocess, sys
    subprocess.check_call([sys.executable, f"{REPO_DIR}/gen_split_idx.py"])
    print("  Done.")
else:
    print(f"split_idx.npz already present: {SPLIT_IDX_PATH}")


checks = {
    "point_cloud.ply":  f"{TRAINED_MODEL_PATH}/point_cloud.ply",
    "cams.npz":         f"{TRAINED_MODEL_PATH}/cams.npz",
    "shadow_net.pt":    f"{TRAINED_MODEL_PATH}/shadow_net.pt",
    "verts_offset.npy": f"{TRAINED_MODEL_PATH}/verts_offset.npy",
    "params_460.npz":   f"{DATASET_DIR}/params_460.npz",
    "Wan low_noise":    f"{WAN_CKPT_DIR}/low_noise_model",
    "Wan high_noise":   f"{WAN_CKPT_DIR}/high_noise_model",
    "Wan VAE":          f"{WAN_CKPT_DIR}/Wan2.1_VAE.pth",
    "Wan T5":           f"{WAN_CKPT_DIR}/models_t5_umt5-xxl-enc-bf16.pth",
}
all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"{'✓' if ok else '✗'} {name}")
print()
print("✓ Ready to run" if all_ok else "✗ Fix missing files")

## Cell 10 — WandB Login (optional)

In [ ]:
if USE_WANDB:
    import wandb
    if WANDB_API_KEY:
        wandb.login(key=WANDB_API_KEY)
    else:
        wandb.login()  # interactive prompt
    print("WandB logged in")
else:
    import os
    os.environ['WANDB_MODE'] = 'disabled'
    print("WandB disabled")

## Cell 11 — Verify Everything Before Running

In [ ]:
import os

checks = {
    "Repo":                 REPO_DIR,
    "train_sds_physics.py": f"{REPO_DIR}/train_sds_physics.py",
    "bridge_sds/wan22":     f"{REPO_DIR}/bridge_sds/wan22_i2v_guidance.py",
    "SDS config yaml":      SDS_CFG,
    "Gaussians dir":        TRAINED_MODEL_PATH,
    "Wan ckpt dir":         WAN_CKPT_DIR,
    "Wan repo":             WAN_REPO_ROOT,
    "Dataset dir":          DATASET_DIR,
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    if not exists:
        all_ok = False
    print(f"{'✓' if exists else '✗ MISSING'}  {name}: {path}")

print()
print("✓ All checks passed — ready to run" if all_ok else "✗ Fix missing paths above before running")

## Cell 12 — Run SDS Physics Training

In [ ]:
import os
import sys

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, WAN_REPO_ROOT)

# Set tokenizer cache so it finds the pre-downloaded T5
os.environ['TRANSFORMERS_CACHE'] = T5_CACHE
os.environ['HF_HOME']            = T5_CACHE

wandb_flag    = "--use_wandb" if USE_WANDB else ""
split_idx_flag = f"--split_idx_path {SPLIT_IDX_PATH}" if SPLIT_IDX_PATH else ""

cmd = f"""
python train_sds_physics.py \\
    --trained_model_path {TRAINED_MODEL_PATH} \\
    --model_path         {MODEL_PATH} \\
    --dataset_dir        {DATASET_DIR} \\
    {split_idx_flag} \\
    --output_dir         {OUTPUT_DIR} \\
    --actor              {ACTOR} \\
    --sequence           {SEQUENCE} \\
    --train_frame_start_num {TRAIN_FRAME_START} {TRAIN_FRAME_NUM} \\
    --verts_start_idx    {VERTS_START_IDX} \\
    --wan_ckpt_dir       {WAN_CKPT_DIR} \\
    --wan_repo_root      {WAN_REPO_ROOT} \\
    --sds_cfg            {SDS_CFG} \\
    --iterations         {ITERATIONS} \\
    --save_name          sds_test \\
    {wandb_flag} \\
    --wandb_project      {WANDB_PROJECT}
""".strip()

print("Running command:")
print(cmd)
print("\n" + "="*60 + "\n")

os.system(cmd)

## Cell 13 — Monitor Output Files

In [ ]:
import os
import pandas as pd

# Find param trajectory CSV
csv_candidates = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        if f == "param_trajectory.csv":
            csv_candidates.append(os.path.join(root, f))

if csv_candidates:
    csv_path = csv_candidates[-1]
    print(f"Found: {csv_path}")
    df = pd.read_csv(csv_path)
    print(df.tail(20).to_string())
else:
    print("No param_trajectory.csv found yet")

## Cell 14 — Plot Parameter Trajectories

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if csv_candidates:
    df = pd.read_csv(csv_candidates[-1])

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("SDS Physics Parameter Optimization", fontsize=14)

    # Parameters
    axes[0,0].plot(df['step'], df['D'],        'b-o', ms=3); axes[0,0].set_title('Density D')
    axes[0,1].plot(df['step'], df['E_Pa'],     'r-o', ms=3); axes[0,1].set_title('Young\'s E (Pa)')
    axes[0,2].plot(df['step'], df['H'],        'g-o', ms=3); axes[0,2].set_title('Height Scale H')
    axes[1,0].plot(df['step'], df['friction'], 'm-o', ms=3); axes[1,0].set_title('Friction')

    # Loss
    axes[1,1].plot(df['step'], df['sds_loss_base'], 'k-o', ms=3); axes[1,1].set_title('SDS Loss (base)')

    # Gradients
    axes[1,2].plot(df['step'], df['grad_D'], label='D')
    axes[1,2].plot(df['step'], df['grad_E'], label='E')
    axes[1,2].plot(df['step'], df['grad_H'], label='H')
    axes[1,2].set_title('SPSA Gradients')
    axes[1,2].legend()

    for ax in axes.flat:
        ax.set_xlabel('step')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/param_trajectory.png", dpi=150)
    plt.show()
    print(f"Saved to {OUTPUT_DIR}/param_trajectory.png")
else:
    print("No CSV data yet")

## Cell 15 — Show Best Params Found

In [ ]:
import numpy as np
import glob

npz_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/sds_best_param_*.npz", recursive=True))

if npz_files:
    latest = npz_files[-1]
    print(f"Latest checkpoint: {latest}")
    data = np.load(latest)
    print("\nBest parameters found:")
    for k, v in data.items():
        print(f"  {k:12s} = {float(v):.6f}")
else:
    print("No checkpoint files found yet")